# 🏆 Numerai Combined Submission Script

This notebook trains and submits predictions for **all three models**:
- **LightGBM** → `jewellzilla_std`
- **CatBoost** → `jewellzilla_cat`  
- **XGBoost** → `jewellzilla_xg`

Run all cells to complete the full pipeline.

In [1]:
#!/usr/bin/env python3
"""
Numerai Competition - Combined Submission Script
Trains LightGBM, CatBoost, and XGBoost models and submits to Numerai
"""

import pandas as pd
import numpy as np
import gc
import sys
from pathlib import Path
from datetime import datetime
import time

from numerapi import NumerAPI
import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

# ============================================================
# CONFIGURATION
# ============================================================

# API Credentials
PUBLIC_ID = 'KY2YNIU7TAALGQRVVTHERFRG7QUJP3FJ'
SECRET_KEY = 'EIMJDP62GSPHETVXXZGLFLOOGKFBRBA6DYPJ7WWSVRMUR3OFKGPIOVVZYHWXBHP4'

# Model IDs
MODEL_IDS = {
    'lightgbm': 'bd2f8540-d90a-4206-b1c5-4e28f2865cba',   # jewellzilla_std
    'catboost': '9e253cd6-6b6b-4178-a641-c9738f21eb11',   # jewellzilla_cat
    'xgboost': 'a65acf61-b5ba-4982-a7c8-7339be001a13'     # jewellzilla_xg
}

# Data Configuration
DATA_VERSION = "v5.2"
TRAINING_FILE = f"{DATA_VERSION}/train.parquet"
LIVE_FILE = f"{DATA_VERSION}/live.parquet"

# Memory Saving Settings
SAMPLE_FRACTION = 0.10  # Use 10% of training data
MAX_FEATURES = 1000     # Limit features

# Date for filenames
DATE_STR = datetime.now().strftime("%Y%m%d")

# Submissions folder
SUBMISSIONS_DIR = Path("submissions")
SUBMISSIONS_DIR.mkdir(exist_ok=True)

print("="*70)
print("🏆 NUMERAI COMBINED SUBMISSION SCRIPT")
print("="*70)
print(f"\n📅 Date: {DATE_STR}")
print(f"💾 Sample fraction: {SAMPLE_FRACTION*100}%")
print(f"📊 Max features: {MAX_FEATURES}")
print(f"\n🎯 Models to train and submit:")
print(f"   • LightGBM → jewellzilla_std")
print(f"   • CatBoost → jewellzilla_cat")
print(f"   • XGBoost  → jewellzilla_xg")

🏆 NUMERAI COMBINED SUBMISSION SCRIPT

📅 Date: 20251224
💾 Sample fraction: 10.0%
📊 Max features: 1000

🎯 Models to train and submit:
   • LightGBM → jewellzilla_std
   • CatBoost → jewellzilla_cat
   • XGBoost  → jewellzilla_xg


In [2]:
# ============================================================
# ROBUST DOWNLOAD FUNCTION (with resume support)
# ============================================================

import requests
from tqdm import tqdm
import os

def download_with_retry(napi, filename, max_retries=10, chunk_size=1024*1024):
    """
    Download a Numerai dataset with retry and resume support.
    Automatically resumes from where it left off if connection drops.
    """
    dest_path = Path(filename)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Get the download URL from Numerai API
    query = "query($filename: String!) { dataset(filename: $filename) }"
    args = {'filename': filename}
    url = napi.raw_query(query, args)['data']['dataset']
    
    print(f"\n📥 Downloading {filename}...")
    
    for attempt in range(max_retries):
        try:
            # Check if partial download exists
            resume_pos = 0
            mode = 'wb'  # Write mode
            
            if dest_path.exists():
                resume_pos = dest_path.stat().st_size
                mode = 'ab'  # Append mode (resume)
                print(f"   📂 Resuming from {resume_pos / (1024**3):.2f} GB...")
            else:
                print(f"   🆕 Starting fresh download...")
            
            # Set up headers for resume
            headers = {}
            if resume_pos > 0:
                headers['Range'] = f'bytes={resume_pos}-'
            
            # Start download
            response = requests.get(url, headers=headers, stream=True, timeout=30)
            
            # Get total size
            if 'content-range' in response.headers:
                total_size = int(response.headers['content-range'].split('/')[-1])
            else:
                total_size = int(response.headers.get('content-length', 0)) + resume_pos
            
            print(f"   📦 Total size: {total_size / (1024**3):.2f} GB")
            print(f"   ⬇️  Attempt {attempt + 1}/{max_retries}...")
            
            # Download with progress bar
            with open(dest_path, mode) as f:
                with tqdm(total=total_size, initial=resume_pos, unit='B', 
                          unit_scale=True, desc=filename.split('/')[-1]) as pbar:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            
            # Verify download completed
            final_size = dest_path.stat().st_size
            if final_size >= total_size:
                print(f"   ✅ Download complete!")
                
                # Verify it's valid parquet
                with open(dest_path, 'rb') as f:
                    first = f.read(4)
                    f.seek(-4, 2)
                    last = f.read()
                
                if first == b'PAR1' and last == b'PAR1':
                    print(f"   ✅ Valid parquet file")
                    return True
                else:
                    print(f"   ⚠️  Invalid parquet file, retrying...")
                    os.remove(dest_path)
                    continue
            else:
                print(f"   ⚠️  Incomplete ({final_size}/{total_size}), retrying...")
                
        except Exception as e:
            error_msg = str(e)[:80]
            print(f"   ⚠️  Attempt {attempt + 1}/{max_retries} failed: {error_msg}")
            if attempt < max_retries - 1:
                wait_time = 5
                print(f"   ⏳ Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"   ❌ Failed after {max_retries} attempts")
                print(f"   💡 File will resume from {dest_path.stat().st_size / (1024**3):.2f} GB if you re-run")
                raise Exception(f"Failed to download {filename} after {max_retries} attempts")
    
    return False

print("✅ Robust download function ready (with auto-resume support)")

✅ Robust download function ready (with auto-resume support)


In [3]:
# ============================================================
# STEP 1: Connect to Numerai API
# ============================================================

print("\n" + "="*70)
print("📡 STEP 1: Connecting to Numerai API")
print("="*70)

napi = NumerAPI(PUBLIC_ID, SECRET_KEY)

try:
    account = napi.get_account()
    print(f"\n✅ Connected as: {account.get('username', 'Unknown')}")
    
    models = napi.get_models()
    print(f"✅ Available models: {list(models.keys())}")
    
    current_round = napi.get_current_round()
    print(f"✅ Current round: {current_round}")
except Exception as e:
    print(f"❌ API connection failed: {e}")
    raise


📡 STEP 1: Connecting to Numerai API

✅ Connected as: jewellzilla
✅ Available models: ['jewellzilla_std', 'jewellzilla_cat', 'jewellzilla_xg']
✅ Current round: 1166


In [4]:
# ============================================================
# STEP 2: Download Data (with retry support)
# ============================================================

print("\n" + "="*70)
print("📥 STEP 2: Downloading Data")
print("="*70)

# Get current round number
current_round = napi.get_current_round()
print(f"\n🎯 Current round: {current_round}")

# List available files to see what's there
print(f"\n📂 Checking available {DATA_VERSION} files...")
available_files = napi.list_datasets()
v52_files = [f for f in available_files if f.startswith(DATA_VERSION)]
live_files = [f for f in v52_files if 'live' in f]
print(f"   Available live files: {live_files}")

# Download training data if not exists (or incomplete)
training_complete = Path(TRAINING_FILE).exists() and Path(TRAINING_FILE).stat().st_size > 2_000_000_000

if not training_complete:
    print(f"\n⬇️  Downloading {TRAINING_FILE} (this may take a while)...")
    download_with_retry(napi, TRAINING_FILE)
else:
    print(f"\nℹ️  {TRAINING_FILE} already exists ({Path(TRAINING_FILE).stat().st_size / 1e9:.2f} GB)")

# Try to download live data - check what's available
LIVE_FILE_ROUND = f"{DATA_VERSION}/live_{current_round}.parquet"
LIVE_FILE_GENERIC = f"{DATA_VERSION}/live.parquet"

# Determine which live file to use
if LIVE_FILE_ROUND in available_files:
    LIVE_FILE = LIVE_FILE_ROUND
    print(f"\n⬇️  Downloading round-specific: {LIVE_FILE}")
elif LIVE_FILE_GENERIC in available_files:
    LIVE_FILE = LIVE_FILE_GENERIC
    print(f"\n⬇️  Downloading generic live file: {LIVE_FILE}")
else:
    # Find any live file for this version
    if live_files:
        LIVE_FILE = live_files[0]
        print(f"\n⬇️  Downloading available live file: {LIVE_FILE}")
    else:
        raise ValueError(f"No live data files found for {DATA_VERSION}!")

download_with_retry(napi, LIVE_FILE)
print(f"\n✅ Live data ready: {LIVE_FILE}")


📥 STEP 2: Downloading Data

🎯 Current round: 1166

📂 Checking available v5.2 files...
   Available live files: ['v5.2/live.parquet', 'v5.2/live_benchmark_models.parquet', 'v5.2/live_example_preds.csv', 'v5.2/live_example_preds.parquet']

ℹ️  v5.2/train.parquet already exists (2.57 GB)

⬇️  Downloading generic live file: v5.2/live.parquet

📥 Downloading v5.2/live.parquet...
   📂 Resuming from 0.01 GB...
   📦 Total size: 0.01 GB
   ⬇️  Attempt 1/10...


live.parquet: 9.53MB [00:00, 379kB/s]                                                                                  

   ✅ Download complete!
   ⚠️  Invalid parquet file, retrying...
   🆕 Starting fresh download...


   📦 Total size: 0.01 GB
   ⬇️  Attempt 2/10...


live.parquet: 100%|███████████████████████████████████████████████████████████████| 9.53M/9.53M [00:06<00:00, 1.59MB/s]

   ✅ Download complete!
   ✅ Valid parquet file

✅ Live data ready: v5.2/live.parquet


In [5]:
# ============================================================
# STEP 3: Load and Prepare Training Data (MEMORY EFFICIENT)
# ============================================================

print("\n" + "="*70)
print("📊 STEP 3: Loading and Preparing Data (Memory-Efficient Mode)")
print("="*70)

import pyarrow.parquet as pq

start_time = datetime.now()

# First, get the column names without loading data
print(f"\n🔄 Reading column names from {TRAINING_FILE}...")
parquet_file = pq.ParquetFile(TRAINING_FILE)
all_columns = parquet_file.schema.names

# Get feature columns (limited to MAX_FEATURES)
all_features = [col for col in all_columns if col.startswith("feature_")]
feature_cols = all_features[:MAX_FEATURES]
print(f"   Found {len(all_features):,} features, using {len(feature_cols):,}")

# Columns to load: selected features + target + id
cols_to_load = feature_cols + ["target"]

# Calculate how many rows to load based on sample fraction
total_rows = parquet_file.metadata.num_rows
sample_size = int(total_rows * SAMPLE_FRACTION)
print(f"   Total rows: {total_rows:,}, sampling {sample_size:,} ({SAMPLE_FRACTION*100}%)")

# Load only the columns we need
print(f"\n🔄 Loading {len(cols_to_load)} columns...")
training_data = pd.read_parquet(
    TRAINING_FILE,
    columns=cols_to_load
)

# Sample immediately to free memory
print(f"\n🔄 Sampling data...")
training_data = training_data.sample(n=sample_size, random_state=42)
gc.collect()

print(f"   ✅ Loaded {len(training_data):,} rows")
print(f"   💾 Memory: {training_data.memory_usage(deep=True).sum() / (1024**2):.0f} MB")

# Prepare X and y
X_train = training_data[feature_cols]
y_train = training_data["target"]

# Free the original dataframe
del training_data
gc.collect()

# Validate
assert len(X_train) > 0, "No training data!"
assert X_train.isnull().sum().sum() == 0, "Missing values in features!"

print(f"\n📊 Data Statistics:")
print(f"   X_train shape: {X_train.shape}")
print(f"   Target mean:   {y_train.mean():.6f}")
print(f"   Target std:    {y_train.std():.6f}")

load_time = (datetime.now() - start_time).total_seconds()
print(f"\n⏱️  Data loading completed in {load_time:.1f}s")


📊 STEP 3: Loading and Preparing Data (Memory-Efficient Mode)

🔄 Reading column names from v5.2/train.parquet...
   Found 2,748 features, using 1,000
   Total rows: 2,746,268, sampling 274,626 (10.0%)

🔄 Loading 1001 columns...

🔄 Sampling data...
   ✅ Loaded 274,626 rows
   💾 Memory: 280 MB

📊 Data Statistics:
   X_train shape: (274626, 1000)
   Target mean:   0.499878
   Target std:    0.223926

⏱️  Data loading completed in 19.7s


In [6]:
# Fix corrupted live.parquet
import os
from pathlib import Path

LIVE_FILE = "v5.2/live.parquet"

print("🔍 Checking live.parquet...")

if Path(LIVE_FILE).exists():
    size_mb = Path(LIVE_FILE).stat().st_size / (1024**2)
    print(f"   Current size: {size_mb:.2f} MB")
    
    # Check if it's valid
    try:
        test = pd.read_parquet(LIVE_FILE)
        print(f"   ✅ File is OK: {test.shape}")
    except:
        print(f"   ❌ File is corrupted!")
        print(f"   🗑️  Deleting...")
        os.remove(LIVE_FILE)
        
        print(f"   📥 Re-downloading...")
        download_with_retry(napi, LIVE_FILE, max_retries=5)
        
        print(f"   ✅ Fixed!")

🔍 Checking live.parquet...
   Current size: 9.09 MB
   ✅ File is OK: (6707, 2791)


In [7]:
# ============================================================
# Load Live Data (used for all predictions)
# ============================================================

print("\n🔄 Loading live data...")
live_data = pd.read_parquet(LIVE_FILE)
print(f"   ✅ Live data shape: {live_data.shape}")


🔄 Loading live data...
   ✅ Live data shape: (6707, 2791)


In [8]:
# ============================================================
# Load Validation Data (for overfitting detection)
# ============================================================

VALIDATION_FILE = f"{DATA_VERSION}/validation.parquet"

print("\n🔄 Loading validation data...")
import time
start = time.time()

try:
    # Check if validation file exists
    if not Path(VALIDATION_FILE).exists():
        print(f"   ⚠️  Validation file not found, downloading...")
        download_with_retry(napi, VALIDATION_FILE)
    
    print(f"   📂 Reading parquet file...")
    # Load validation data (sample same as training)
    val_sample_size = int(574882 * SAMPLE_FRACTION)  # ~575k rows in validation
    
    # THIS IS THE KEY - print before the slow operation
    print(f"   ⏳ Loading {val_sample_size:,} rows (this may take 2-3 minutes)...")
    print(f"   💡 Browser may show 'connection lost' - this is normal, kernel is still running")
    
    validation_data = pd.read_parquet(VALIDATION_FILE)
    print(f"   ✅ Loaded full file ({len(validation_data):,} rows)")
    
    print(f"   🔄 Sampling {val_sample_size:,} rows...")
    validation_data = validation_data.sample(n=val_sample_size, random_state=42)
    
    X_val = validation_data[feature_cols]
    y_val = validation_data["target"]
    
    # Free memory
    del validation_data
    gc.collect()
    
    elapsed = time.time() - start
    print(f"   ✅ Validation data loaded: {len(X_val):,} rows ({elapsed:.1f}s)")
    validation_available = True
    
except Exception as e:
    print(f"   ⚠️  Could not load validation data: {e}")
    print(f"   Continuing without validation...")
    validation_available = False
    X_val, y_val = None, None


🔄 Loading validation data...
   📂 Reading parquet file...
   ⏳ Loading 57,488 rows (this may take 2-3 minutes)...
   💡 Browser may show 'connection lost' - this is normal, kernel is still running
   ✅ Loaded full file (3,882,191 rows)
   🔄 Sampling 57,488 rows...
   ✅ Validation data loaded: 57,488 rows (163.0s)


In [9]:
# ============================================================# STEP 4A: Train LightGBM Model# ============================================================print("\n" + "="*70)print("🌲 STEP 4A: Training LightGBM Model")print("="*70)lgb_params = {    'n_estimators': 1000,    'learning_rate': 0.05,    'max_depth': 4,    'num_leaves': 16,    'colsample_bytree': 0.1,    'random_state': 42,    'n_jobs': -1,    'verbose': -1}print("\n🔄 Training LightGBM...")start = datetime.now()lgb_model = lgb.LGBMRegressor(**lgb_params)lgb_model.fit(X_train, y_train)train_time = (datetime.now() - start).total_seconds()print(f"   ✅ Training completed in {train_time:.1f}s")# Training correlationlgb_train_preds = lgb_model.predict(X_train)lgb_train_corr = np.corrcoef(lgb_train_preds, y_train)[0, 1]print(f"   📈 Training correlation: {lgb_train_corr:.4f}")# === VALIDATION CHECK ===if validation_available:    print(f"\n   📊 Validating on out-of-sample data...")    lgb_val_preds = lgb_model.predict(X_val)    lgb_val_corr = np.corrcoef(lgb_val_preds, y_val)[0, 1]    print(f"   📊 Validation correlation: {lgb_val_corr:.4f}")        diff = lgb_train_corr - lgb_val_corr    if diff > 0.05:        print(f"   ⚠️  WARNING: Overfitting detected! (diff: {diff:.4f})")    elif diff > 0.02:        print(f"   ⚠️  Mild overfitting (diff: {diff:.4f})")    else:        print(f"   ✅ Model generalizes well!")# === END VALIDATION ===# Generate predictionsprint("\n🔮 Generating predictions...")lgb_predictions = lgb_model.predict(live_data[feature_cols])# Normalize to [0, 1]lgb_predictions_norm = (lgb_predictions - lgb_predictions.min()) / (lgb_predictions.max() - lgb_predictions.min())# Save submissionlgb_filename = SUBMISSIONS_DIR / f"submission_lightgbm_{DATE_STR}.csv"lgb_submission = pd.DataFrame({    "id": live_data.index,    "prediction": lgb_predictions_norm})lgb_submission.to_csv(lgb_filename, index=False)print(f"   ✅ Saved {len(lgb_submission):,} predictions to {lgb_filename}")print(f"   📊 Mean: {lgb_predictions_norm.mean():.6f}")

In [10]:
# ============================================================# STEP 4B: Train CatBoost Model# ============================================================print("\n" + "="*70)print("🐱 STEP 4B: Training CatBoost Model")print("="*70)print("\n🔄 Training CatBoost...")start = datetime.now()cat_model = CatBoostRegressor(    iterations=1000,    learning_rate=0.05,    depth=4,    verbose=False)cat_model.fit(X_train, y_train)train_time = (datetime.now() - start).total_seconds()print(f"   ✅ Training completed in {train_time:.1f}s")# Training correlationcat_train_preds = cat_model.predict(X_train)cat_train_corr = np.corrcoef(cat_train_preds, y_train)[0, 1]print(f"   📈 Training correlation: {cat_train_corr:.4f}")# === VALIDATION CHECK ===if validation_available:    print(f"\n   📊 Validating on out-of-sample data...")    cat_val_preds = cat_model.predict(X_val)    cat_val_corr = np.corrcoef(cat_val_preds, y_val)[0, 1]    print(f"   📊 Validation correlation: {cat_val_corr:.4f}")        diff = cat_train_corr - cat_val_corr    if diff > 0.05:        print(f"   ⚠️  WARNING: Overfitting detected! (diff: {diff:.4f})")    elif diff > 0.02:        print(f"   ⚠️  Mild overfitting (diff: {diff:.4f})")    else:        print(f"   ✅ Model generalizes well!")# === END VALIDATION ===# Generate predictionsprint("\n🔮 Generating predictions...")cat_predictions = cat_model.predict(live_data[feature_cols])# Normalize to [0, 1]cat_predictions_norm = (cat_predictions - cat_predictions.min()) / (cat_predictions.max() - cat_predictions.min())# Save submissioncat_filename = SUBMISSIONS_DIR / f"submission_catboost_{DATE_STR}.csv"cat_submission = pd.DataFrame({    "id": live_data.index,    "prediction": cat_predictions_norm})cat_submission.to_csv(cat_filename, index=False)print(f"   ✅ Saved {len(cat_submission):,} predictions to {cat_filename}")print(f"   📊 Mean: {cat_predictions_norm.mean():.6f}")

In [11]:
# ============================================================# STEP 4C: Train XGBoost Model# ============================================================print("\n" + "="*70)print("🚀 STEP 4C: Training XGBoost Model")print("="*70)print("\n🔄 Training XGBoost...")start = datetime.now()xgb_model = XGBRegressor(    n_estimators=1000,    learning_rate=0.05,    max_depth=4,    colsample_bytree=0.1,    random_state=42,    verbosity=0)xgb_model.fit(X_train, y_train)train_time = (datetime.now() - start).total_seconds()print(f"   ✅ Training completed in {train_time:.1f}s")# Training correlationxgb_train_preds = xgb_model.predict(X_train)xgb_train_corr = np.corrcoef(xgb_train_preds, y_train)[0, 1]print(f"   📈 Training correlation: {xgb_train_corr:.4f}")# === VALIDATION CHECK ===if validation_available:    print(f"\n   📊 Validating on out-of-sample data...")    xgb_val_preds = xgb_model.predict(X_val)    xgb_val_corr = np.corrcoef(xgb_val_preds, y_val)[0, 1]    print(f"   📊 Validation correlation: {xgb_val_corr:.4f}")        diff = xgb_train_corr - xgb_val_corr    if diff > 0.05:        print(f"   ⚠️  WARNING: Overfitting detected! (diff: {diff:.4f})")    elif diff > 0.02:        print(f"   ⚠️  Mild overfitting (diff: {diff:.4f})")    else:        print(f"   ✅ Model generalizes well!")# === END VALIDATION ===# Generate predictionsprint("\n🔮 Generating predictions...")xgb_predictions = xgb_model.predict(live_data[feature_cols])# Normalize to [0, 1]xgb_predictions_norm = (xgb_predictions - xgb_predictions.min()) / (xgb_predictions.max() - xgb_predictions.min())# Save submissionxgb_filename = SUBMISSIONS_DIR / f"submission_xgboost_{DATE_STR}.csv"xgb_submission = pd.DataFrame({    "id": live_data.index,    "prediction": xgb_predictions_norm})xgb_submission.to_csv(xgb_filename, index=False)print(f"   ✅ Saved {len(xgb_submission):,} predictions to {xgb_filename}")print(f"   📊 Mean: {xgb_predictions_norm.mean():.6f}")

In [12]:
# ============================================================
# Training Summary
# ============================================================

print("\n" + "="*70)
print("📊 TRAINING SUMMARY")
print("="*70)

print(f"\n{'Model':<12} {'Train Corr':<12} {'Pred Mean':<12} {'File'}")
print("-"*60)
print(f"{'LightGBM':<12} {lgb_train_corr:<12.4f} {lgb_predictions_norm.mean():<12.6f} {lgb_filename}")
print(f"{'CatBoost':<12} {cat_train_corr:<12.4f} {cat_predictions_norm.mean():<12.6f} {cat_filename}")
print(f"{'XGBoost':<12} {xgb_train_corr:<12.4f} {xgb_predictions_norm.mean():<12.6f} {xgb_filename}")

print("\n✅ All models trained and predictions saved!")


📊 TRAINING SUMMARY

Model        Train Corr   Pred Mean    File
------------------------------------------------------------


NameError: name 'lgb_train_corr' is not defined

In [ ]:
# ============================================================
# STEP 5: Submit All Predictions to Numerai
# ============================================================

print("\n" + "="*70)
print("🚀 STEP 5: Submitting All Predictions to Numerai")
print("="*70)

submissions = [
    ('LightGBM', 'jewellzilla_std', MODEL_IDS['lightgbm'], lgb_filename),
    ('CatBoost', 'jewellzilla_cat', MODEL_IDS['catboost'], cat_filename),
    ('XGBoost', 'jewellzilla_xg', MODEL_IDS['xgboost'], xgb_filename)
]

results = {}

for model_name, numerai_name, model_id, filename in submissions:
    print(f"\n{'='*50}")
    print(f"📤 Submitting {model_name} → {numerai_name}")
    print(f"{'='*50}")
    print(f"   Model ID: {model_id}")
    print(f"   File: {filename}")
    
    try:
        submission_id = napi.upload_predictions(filename, model_id=model_id)
        print(f"   ✅ SUCCESS!")
        print(f"   Submission ID: {submission_id}")
        results[model_name] = {'status': 'SUCCESS', 'id': submission_id}
        time.sleep(2)  # Small delay between submissions
    except Exception as e:
        print(f"   ❌ FAILED: {e}")
        results[model_name] = {'status': 'FAILED', 'error': str(e)}

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*70)
print("🏆 FINAL SUBMISSION SUMMARY")
print("="*70)

print(f"\n⏰ Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"👤 Account: {account.get('username', 'Unknown')}")
print(f"🎯 Round: {current_round}")

print(f"\n{'Model':<12} {'Numerai Model':<18} {'Status':<10} {'Submission ID'}")
print("-"*70)

success_count = 0
for model_name, numerai_name, model_id, filename in submissions:
    result = results.get(model_name, {})
    status = result.get('status', 'UNKNOWN')
    sub_id = result.get('id', result.get('error', 'N/A'))[:36] if result else 'N/A'
    
    status_icon = '✅' if status == 'SUCCESS' else '❌'
    print(f"{model_name:<12} {numerai_name:<18} {status_icon} {status:<8} {sub_id}")
    
    if status == 'SUCCESS':
        success_count += 1

print("\n" + "="*70)

if success_count == 3:
    print("🎉 ALL 3 MODELS SUBMITTED SUCCESSFULLY!")
    print("\n💡 What's Next:")
    print("   • Check dashboard: https://numer.ai/models")
    print("   • Scores appear in 2-3 days")
    print("   • Full evaluation takes 20 days")
    print("   • Compare performance between models!")
elif success_count > 0:
    print(f"⚠️  PARTIAL SUCCESS: {success_count}/3 models submitted")
    print("   Review errors above and retry failed submissions")
else:
    print("❌ ALL SUBMISSIONS FAILED")
    print("   Check API credentials and try again")

print("\n" + "="*70)
print("✅ SCRIPT COMPLETE")
print("="*70)